In [0]:
%pip install FlagEmbedding

In [0]:
# ============================================================================
# TOPIC MODELING NOTEBOOK — Cell 1: Setup + load from deviation_embeddings
#
# SOURCE = deviation_embeddings  (NOT deviation_embed_input) because it is the
# only table holding BOTH the precomputed BGE-M3 vectors AND the text:
#   • BERTopic → reuses the `embedding` (fine, full-context) column → no re-encoding
#   • c-TF-IDF → uses `source_free_text_full` (clean deviation text) → bag-of-words
#
# New contextual-retrieval schema (from build_reference_glossary):
#   embedding                    ← BGE-M3 over contextual_retrieval_text_full
#   core_embedding               ← BGE-M3 over contextual_retrieval_text_core
#   source_free_text_{full,core} ← cleaned deviation text (NO enrichment)
#   contextual_retrieval_text_*  ← deterministic refs + llm context + source
# Clustering uses the enriched `embedding`; labeling uses the clean source text.
# ============================================================================
import numpy as np
from pyspark.sql import functions as F

CATALOG = "us_gmsgq_dev"          # ← match your env
ALYT    = "gms_us_alyt"
EMB_TABLE = f"{CATALOG}.{ALYT}.deviation_embeddings"

pdf = (
    spark.table(EMB_TABLE)
    .select("pr_id", "source_free_text_full", "embedding")
    .toPandas()
)
pdf["source_free_text_full"] = pdf["source_free_text_full"].fillna("")

import re

def _strip_for_tfidf(text):
    """Strip the 'Column_Name: ' scaffolding from source_free_text so c-TF-IDF sees
    only content. The precomputed embeddings (BGE-M3 over the ENRICHED
    contextual_retrieval_text) already encode the resolved-reference context — here
    we only need clean text for the CountVectorizer keyword-labeling step."""
    # Remove column-name prefixes ("Event_Title: ", "Root_Cause_SubCategory: ", etc.)
    text = re.sub(
        r"\b(Event_Title|Event_Description|Impact_Assessment|Quality_Final_Assessment"
        r"|Root_Cause_Category|Root_Cause_SubCategory|Action_Text)\s*:",
        "", text
    )
    return text.strip()

# docs  = clean deviation text for c-TF-IDF labeling only
# embs  = precomputed fine-tier BGE-M3 vectors (drive UMAP + HDBSCAN clustering)
docs = [_strip_for_tfidf(t) for t in pdf["source_free_text_full"].tolist()]
embs = np.array(pdf["embedding"].tolist(), dtype=np.float32)

print(f"Docs: {len(docs):,}  |  Embedding matrix: {embs.shape}")
assert embs.shape[1] == 1024, "expected 1024-dim BGE-M3 vectors"

In [0]:
%pip install bertopic

In [0]:
%pip install umap-learn

In [0]:
# ============================================================================
# Zero-Shot BERTopic — Using Precomputed BGE-M3 Embeddings
#
# Same zero-shot approach as the MiniLM cell, but leverages the PRECOMPUTED
# BGE-M3 vectors (encoded over contextual_retrieval_text_full). This means:
#   • Document embeddings = already computed, no re-encoding needed
#   • Label embeddings   = encoded on-the-fly by BGE-M3 (just 18 short strings)
#   • Both live in the SAME 1024-dim BGE-M3 space → valid cosine similarity
#
# Advantage over MiniLM zero-shot: the doc embeddings capture the full enriched
# context (resolved references, situating context) rather than just the raw
# source_free_text_full, giving richer semantic matching to topic labels.
# ============================================================================
import numpy as np, pandas as pd, plotly.express as px
from FlagEmbedding import BGEM3FlagModel
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

# ---- BGE-M3 wrapper compatible with BERTopic's embedding_model interface ----
from bertopic.backend import BaseEmbedder

class BGEM3Wrapper(BaseEmbedder):
    """Wraps FlagEmbedding's BGEM3FlagModel so BERTopic can use it to encode
    candidate topic labels. Inherits BaseEmbedder so BERTopic calls embed()
    directly without falling back to SentenceTransformerBackend."""
    def __init__(self, model_name="BAAI/bge-m3", use_fp16=True):
        super().__init__()
        self.model = BGEM3FlagModel(model_name, use_fp16=use_fp16)
        self.embedding_dimension = 1024

    def embed(self, documents, verbose=False):
        """Encode documents and return dense vectors (numpy array, 1024-dim)."""
        output = self.model.encode(
            documents,
            batch_size=32,
            max_length=512,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )
        return np.array(output["dense_vecs"], dtype=np.float32)

# ---- Load BGE-M3 (only used to encode the 18 candidate labels) ----
print("Loading BGE-M3 for label encoding...")
bge_wrapper = BGEM3Wrapper("BAAI/bge-m3", use_fp16=True)
print(f"BGE-M3 loaded  |  dim={bge_wrapper.embedding_dimension}")

# ---- Reuse precomputed doc embeddings from Cell 2 ----
# embs = BGE-M3 over contextual_retrieval_text_full (already in memory)
# docs = source_free_text_full stripped for c-TF-IDF labeling
print(f"Reusing precomputed embeddings: {embs.shape}")
print(f"Docs for c-TF-IDF labeling: {len(docs):,}")

# ---- Same candidate topics as the MiniLM zero-shot cell ----
candidate_topics = [
    "Inadvertent unblinding of clinical trial treatment assignment",
    "GMP manufacturing deviation or batch failure",
    "Labeling and packaging error",
    "Audit finding or inspection observation",
    "Stability testing out of specification result",
    "Data integrity or GCP compliance issue",
    "Training non-compliance or qualification gap",
    "CAPA corrective and preventive action overdue",
    "Clinical protocol deviation or non-compliance",
    "Vendor or supplier quality issue",
    "Document control or SOP deviation",
    "Equipment or instrument malfunction or calibration",
    "Environmental monitoring excursion",
    "Product complaint or adverse event reporting",
    "IT system validation or computerized system issue",
    "Material or component quality failure",
    "Cleaning or contamination control deviation",
    "Transportation or cold chain deviation",
]
print(f"\nCandidate topics: {len(candidate_topics)}")

# ---- Fit Zero-Shot BERTopic with precomputed BGE-M3 embeddings ----
BEST_NN, BEST_NC, BEST_MD = 5, 5, 0.0

IGNORE_WORDS = {
    "event", "title", "description", "impact", "assessment", "quality", "final",
    "root", "cause", "category", "subcategory", "action", "text", "other",
    "not", "applicable", "the", "and", "for", "was", "were", "that", "this",
    "with", "from", "but", "are", "has", "have", "been", "will",
}
custom_stops = list(ENGLISH_STOP_WORDS) + list(IGNORE_WORDS)

zs_bge_model = BERTopic(
    embedding_model=bge_wrapper,               # ← encodes LABELS with BGE-M3
    umap_model=UMAP(n_neighbors=BEST_NN, n_components=BEST_NC, min_dist=BEST_MD,
                    metric="cosine", random_state=42),
    hdbscan_model=HDBSCAN(min_cluster_size=10, min_samples=5,
                          metric="euclidean", cluster_selection_method="eom",
                          prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2)),
    zeroshot_topic_list=candidate_topics,       # ← ZERO-SHOT: pre-defined labels
    zeroshot_min_similarity=0.5,               # cosine sim threshold
    calculate_probabilities=True,
    verbose=True,
)

# Pass precomputed BGE-M3 doc embeddings → BERTopic skips doc encoding,
# only encodes the 18 candidate labels via bge_wrapper.encode()
zs_bge_topics, zs_bge_probs = zs_bge_model.fit_transform(docs, embeddings=embs)

# ---- Results ----
zs_bge_info = zs_bge_model.get_topic_info()
n_zs_bge = len(zs_bge_info[zs_bge_info.Topic != -1])
n_out_bge = int((np.array(zs_bge_topics) == -1).sum())
n_assigned_bge = len(zs_bge_topics) - n_out_bge
print(f"\nBGE-M3 Zero-shot results:")
print(f"  Topics found: {n_zs_bge}  |  Assigned: {n_assigned_bge} ({100*n_assigned_bge/len(zs_bge_topics):.1f}%)")
print(f"  Outliers: {n_out_bge} ({100*n_out_bge/len(zs_bge_topics):.1f}%)")

# ---- Summary table ----
zs_bge_summary = []
for _, row in zs_bge_info.iterrows():
    t = row["Topic"]
    if t == -1:
        zs_bge_summary.append({"Topic": -1, "Label": "(Outlier / Unclassified)",
                               "Count": row["Count"], "Keywords": "", "Source": ""})
    else:
        kws = [w for w, _ in zs_bge_model.get_topic(t)[:8]]
        label = row.get("Name", "") or f"Topic {t}"
        if label.startswith(f"{t}_"):
            kws_short = label.split("_")[1:4]
            label = " / ".join(kws_short)
        source = "Zero-shot" if t < len(candidate_topics) else "Emergent (clustered)"
        zs_bge_summary.append({
            "Topic": t, "Label": label,
            "Count": row["Count"],
            "Keywords": ", ".join(kws),
            "Source": source,
        })

zs_bge_df = pd.DataFrame(zs_bge_summary)
print("\n" + "="*95)
print("ZERO-SHOT TOPIC SUMMARY  (BGE-M3 embeddings over contextual_retrieval_text_full)")
print("Docs matched by cosine similarity in the BGE-M3 1024-dim space (threshold=0.5).")
print("Remaining docs clustered via UMAP + HDBSCAN as emergent topics.")
print("="*95)
display(zs_bge_df)

# ---- Comparison with MiniLM zero-shot (if available) ----
try:
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
    ari = adjusted_rand_score(zs_topics, zs_bge_topics)
    nmi = normalized_mutual_info_score(zs_topics, zs_bge_topics)
    print(f"\nAgreement with MiniLM zero-shot:")
    print(f"  Adjusted Rand Index:          {ari:.4f}")
    print(f"  Normalized Mutual Information: {nmi:.4f}")
except NameError:
    print("\n(MiniLM zero-shot not in memory — skipping comparison)")

# ---- 3D scatter ----
umap_3d = UMAP(n_neighbors=BEST_NN, n_components=3, min_dist=BEST_MD,
               metric="cosine", random_state=42)
coords = umap_3d.fit_transform(embs)

topic_labels_map = {}
for _, row in zs_bge_df.iterrows():
    t = int(row["Topic"])
    lbl = row["Label"]
    topic_labels_map[t] = "Outlier" if t == -1 else f"{t}: {lbl[:45]}"

plot_df = pd.DataFrame({
    "x": coords[:, 0], "y": coords[:, 1], "z": coords[:, 2],
    "topic_label": [topic_labels_map.get(t, f"Topic {t}") for t in zs_bge_topics],
    "pr_id": pdf["pr_id"].values,
    "event_description": [d[:250] + ("..." if len(d) > 250 else "") for d in docs],
})

fig = px.scatter_3d(
    plot_df, x="x", y="y", z="z", color="topic_label",
    hover_data={"pr_id": True, "event_description": True, "x": False, "y": False, "z": False},
    title=f"Zero-Shot BERTopic (BGE-M3) — {n_zs_bge} topics, threshold=0.5",
    opacity=0.75, height=700,
)
fig.update_traces(marker_size=3.5)
fig.update_layout(
    legend_title_text="Topic",
    scene=dict(xaxis_title="", yaxis_title="", zaxis_title=""),
)
fig.show()

# ============================================================================
# MULTI-LABEL ASSIGNMENT via approximate_distribution()
# Same sliding-window approach as cell 10, but on the BGE-M3 zero-shot model.
# ============================================================================
print("\n" + "="*95)
print("MULTI-LABEL TOPIC ASSIGNMENT  (BGE-M3 zero-shot + approximate_distribution)")
print("="*95)

zs_bge_distr, _ = zs_bge_model.approximate_distribution(docs, min_similarity=0.01)
print(f"Distribution matrix shape: {zs_bge_distr.shape}  (docs × topics)")

MULTI_THRESHOLD = 0.1

zs_bge_multi = []
for i in range(len(docs)):
    doc_probs = zs_bge_distr[i]
    above = [(t, float(doc_probs[t])) for t in range(len(doc_probs)) if doc_probs[t] >= MULTI_THRESHOLD]
    above.sort(key=lambda x: -x[1])
    topic_ids = [t for t, _ in above]
    topic_probs = [p for _, p in above]
    topic_labels = [topic_labels_map.get(t, f"Topic {t}") for t in topic_ids]
    zs_bge_multi.append({
        "pr_id": pdf["pr_id"].values[i],
        "n_topics": len(topic_ids),
        "primary_topic": zs_bge_topics[i],
        "primary_label": topic_labels_map.get(zs_bge_topics[i], "Outlier"),
        "all_topics": topic_ids,
        "all_labels": topic_labels,
        "all_probs": topic_probs,
    })

zs_bge_multi_df = pd.DataFrame(zs_bge_multi)

print(f"\nThreshold: {MULTI_THRESHOLD}")
print(f"Events with 1 topic:   {(zs_bge_multi_df['n_topics'] == 1).sum()}")
print(f"Events with 2 topics:  {(zs_bge_multi_df['n_topics'] == 2).sum()}")
print(f"Events with 3+ topics: {(zs_bge_multi_df['n_topics'] >= 3).sum()}")
print(f"Events with 0 topics:  {(zs_bge_multi_df['n_topics'] == 0).sum()}  (below threshold for all)")
print(f"Average topics/event:  {zs_bge_multi_df['n_topics'].mean():.2f}")

multi_events = zs_bge_multi_df[zs_bge_multi_df["n_topics"] >= 2].sort_values("n_topics", ascending=False)
print(f"\nTop multi-topic events ({len(multi_events)} events with ≥2 topics):")
display(multi_events[["pr_id", "n_topics", "all_labels", "all_probs"]].head(20))

In [0]:
# ============================================================================
# LLM-Based Topic Assignment  —  ai_query BATCH version (server-side parallel)
#
# Replaces the ThreadPoolExecutor + OpenAI-client loop with a single ai_query
# batch call. Databricks parallelizes/retries/scales it automatically.
#   - Structured output (json_schema) => guaranteed valid JSON
#   - failOnError => false            => one bad row won't kill the run
#   - Hybrid taxonomy: prefer TOPIC_LIST, allow novel primary/secondary topics
#   - Enum-locked root_cause & regulatory_impact
#   - Python post-processing: snap-to-canonical, novel tracking, rollups
# NOTE: needs serverless compute + DBR 15.4 LTS+ (responseFormat). No GPU needed.
# ============================================================================
import json
from collections import Counter
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, IntegerType
)

MODEL = "databricks-gpt-5-4-mini"
MAX_INPUT_CHARS = 3000
OUT_TABLE = None   # e.g. "your_catalog.your_schema.llm_topic_assignments" to persist; else None

# ---- Controlled vocabularies ----
TOPIC_LIST = [
    "Inadvertent unblinding", "GMP manufacturing deviation",
    "Labeling and packaging error", "Audit finding or inspection observation",
    "Stability testing OOS", "Data integrity or GCP issue",
    "Training non-compliance", "CAPA overdue or inadequate",
    "Clinical protocol deviation", "Vendor or supplier quality issue",
    "Document control or SOP deviation", "Equipment or instrument malfunction",
    "Environmental monitoring excursion", "Product complaint or adverse event",
    "IT system or CSV issue", "Material or component failure",
    "Cleaning or contamination control", "Transportation or cold chain deviation",
    "Pharmacovigilance or safety reporting", "Regulatory submission or labeling update",
    "Clinical study operations",
]
ROOT_CAUSE_LIST = [
    "Human Error", "Training", "Process/Procedure", "Vendor/CRO",
    "System/Technology (IRT/eCRF)", "Documentation", "Clinical Operations",
    "Laboratory", "Manufacturing/Supply", "Not determinable", "Other",
]
IMPACT_LIST = [
    "Patient Safety", "Data Integrity", "Trial Blinding/Unblinding",
    "GCP Compliance", "Product Quality", "Supply Chain",
    "Documentation/Records", "Not determinable", "None",
]

TOPIC_LIST_STR = "\n".join(f"  {i+1}. {t}" for i, t in enumerate(TOPIC_LIST))
ROOT_CAUSE_STR = " | ".join(ROOT_CAUSE_LIST)
IMPACT_STR     = " | ".join(IMPACT_LIST)

# ---- Prompt (system + instructions merged into one text block for ai_query) ----
PROMPT_PREFIX = f"""You are a GxP pharmaceutical quality deviation classifier at Takeda,
specializing in clinical trial deviations and QMS records. Prefer the provided topic
list, but you MAY create a new concise topic (2-5 words) when no listed topic fits.
Work ONLY from evidence in the text; if information is not stated, use "Not determinable"
rather than guessing. Do not invent facts, systems, or root causes.

STEP 1 - PRIMARY topic: match one TOPIC below, else CREATE a concise new label.
STEP 2 - 0-2 SECONDARY topics (listed or newly created, same rule).
STEP 3 - root_cause_category from ROOT CAUSE list.
STEP 4 - regulatory_impact from IMPACT list.
STEP 5 - Quote a short evidence_span justifying the primary topic.
STEP 6 - Score confidence: 90-100 explicit, 70-89 strong implication,
         40-69 partial/indirect, <40 weak/inferred.
STEP 7 - List any topics you CREATED (not in the list) in novel_topics.
Newly created labels must be general business themes, NOT product/study names.

TOPICS:
{TOPIC_LIST_STR}

ROOT CAUSE: {ROOT_CAUSE_STR}

IMPACT: {IMPACT_STR}

Deviation text:
"""

# ---- Structured-output schema (strict json_schema) ----
RESPONSE_FORMAT = json.dumps({
    "type": "json_schema",
    "json_schema": {
        "name": "deviation_topic",
        "schema": {
            "type": "object",
            "properties": {
                "primary":             {"type": "string"},
                "secondary":           {"type": "array", "items": {"type": "string"}},
                "novel_topics":        {"type": "array", "items": {"type": "string"}},
                "root_cause_category": {"type": "string", "enum": ROOT_CAUSE_LIST},
                "regulatory_impact":   {"type": "string", "enum": IMPACT_LIST},
                "evidence_span":       {"type": "string"},
                "confidence":          {"type": "integer"},
                "reasoning":           {"type": "string"},
            },
            "required": ["primary", "secondary", "novel_topics",
                         "root_cause_category", "regulatory_impact",
                         "evidence_span", "confidence", "reasoning"],
            "additionalProperties": False,
        },
        "strict": True,
    },
})

# ---- Build the ai_query expression (single-quoted JSON is safe: no single quotes inside) ----
AI_EXPR = f"""ai_query(
    '{MODEL}',
    prompt,
    responseFormat => '{RESPONSE_FORMAT}',
    modelParameters => named_struct('temperature', 0.0, 'max_tokens', 500),
    failOnError => false
)"""

# ---- Run batch inference (WHOLE table in one query -> server-side parallelism) ----
base = (
    spark.table(EMB_TABLE)
    .select("pr_id", "contextual_retrieval_text_full")
    .withColumn("text_trunc",
                F.substring(F.coalesce("contextual_retrieval_text_full", F.lit("")), 1, MAX_INPUT_CHARS))
    .withColumn("prompt", F.concat(F.lit(PROMPT_PREFIX), F.col("text_trunc")))
    .withColumn("llm_out", F.expr(AI_EXPR))
)

# failOnError=false returns a STRUCT{<response>, errorMessage}. Grab fields robustly.
resp_field = [f for f in base.schema["llm_out"].dataType.fieldNames() if f != "errorMessage"][0]

parse_schema = StructType([
    StructField("primary",             StringType()),
    StructField("secondary",           ArrayType(StringType())),
    StructField("novel_topics",        ArrayType(StringType())),
    StructField("root_cause_category", StringType()),
    StructField("regulatory_impact",   StringType()),
    StructField("evidence_span",       StringType()),
    StructField("confidence",          IntegerType()),
    StructField("reasoning",           StringType()),
])

parsed_sdf = (
    base
    .withColumn("llm_err",  F.col("llm_out.errorMessage"))
    .withColumn("llm_json", F.col(f"llm_out.{resp_field}"))
    .withColumn("p", F.from_json("llm_json", parse_schema))
    .select(
        "pr_id",
        F.col("p.primary").alias("primary"),
        F.col("p.secondary").alias("secondary"),
        F.col("p.novel_topics").alias("novel_topics_model"),
        F.col("p.root_cause_category").alias("root_cause_category"),
        F.col("p.regulatory_impact").alias("regulatory_impact"),
        F.col("p.evidence_span").alias("evidence_span"),
        F.col("p.confidence").alias("confidence"),
        F.col("p.reasoning").alias("reasoning"),
        "llm_err",
    )
)

if OUT_TABLE:
    parsed_sdf.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(OUT_TABLE)
    print(f"Saved -> {OUT_TABLE}")

# ---- Pull to pandas for normalization + rollups ----
df = parsed_sdf.toPandas()
print(f"Documents classified: {len(df):,} | Model: {MODEL} | Topics: {len(TOPIC_LIST)}")

# ---- Snap known topics to canonical spelling; flag novel ones ----
_CANON = {t.lower(): t for t in TOPIC_LIST}

def _norm(label):
    if not isinstance(label, str) or not label.strip():
        return None, False
    key = label.strip().lower()
    return (_CANON[key], True) if key in _CANON else (label.strip(), False)

def _row_norm(row):
    if row["llm_err"] or not isinstance(row["primary"], str):
        return pd.Series({"primary_n": "ERROR", "primary_is_novel": False,
                          "secondary_n": [], "novel_topics": []})
    p_lbl, p_known = _norm(row["primary"])
    sec = [_norm(s) for s in (row["secondary"] if row["secondary"] is not None else [])]
    secondary = [l for l, _ in sec if l]
    novel = ([p_lbl] if (p_lbl and not p_known) else []) + [l for l, k in sec if l and not k]
    novel = sorted(set(novel) | set(row["novel_topics_model"] if row["novel_topics_model"] is not None else []))
    return pd.Series({"primary_n": p_lbl, "primary_is_novel": (p_lbl and not p_known),
                      "secondary_n": secondary, "novel_topics": novel})

df = df.join(df.apply(_row_norm, axis=1))
df["primary"]   = df["primary_n"]
df["secondary"] = df["secondary_n"]
df["n_topics"]  = df.apply(lambda r: 0 if r["primary"] == "ERROR" else 1 + len(r["secondary"]), axis=1)

# ---- Rollups ----
n_err = df["llm_err"].notna().sum()
print(f"Errors: {n_err} ({100*n_err/max(len(df),1):.1f}%)")

def _section(title, series):
    print("\n" + "="*80 + f"\n{title}\n" + "="*80)
    print(series.to_string())

_section("PRIMARY TOPIC DISTRIBUTION", df["primary"].value_counts().head(30))
_section("ROOT CAUSE DISTRIBUTION",    df["root_cause_category"].value_counts())
_section("REGULATORY IMPACT DISTRIBUTION", df["regulatory_impact"].value_counts())

conf = pd.to_numeric(df["confidence"], errors="coerce")
print("\n" + "="*80 + "\nCONFIDENCE\n" + "="*80)
print(f"  mean={conf.mean():.1f}  median={conf.median():.1f}  <40 (needs review): {(conf<40).sum()}")

print("\n" + "="*80 + "\nMULTI-LABEL STATS\n" + "="*80)
for k in [1, 2, 3]:
    print(f"  {k} topic(s): {(df['n_topics']==k).sum()}")
print(f"  4+ topics:  {(df['n_topics']>=4).sum()}")

novel_counter = Counter()
for _, r in df.iterrows():
    novel_counter.update(r["novel_topics"] or [])
print("\n" + "="*80 + "\nNOVEL TOPICS (not in TOPIC_LIST)\n" + "="*80)
print(f"Rows using a novel PRIMARY topic: {df['primary_is_novel'].sum()}")
for topic, cnt in novel_counter.most_common(30):
    print(f"  {cnt:>4}  {topic}")
if not novel_counter:
    print("  (none — every record matched the existing taxonomy)")

valid = (df["primary"] != "ERROR").sum()
known = ((~df["primary_is_novel"]) & (df["primary"] != "ERROR")).sum()
print(f"\nTaxonomy coverage (primary): {known}/{valid} ({100*known/max(valid,1):.1f}%)")

# ---- QA display: low-confidence first ----
display(
    df[["pr_id", "primary", "primary_is_novel", "secondary", "novel_topics",
        "root_cause_category", "regulatory_impact", "confidence", "n_topics",
        "evidence_span", "reasoning", "llm_err"]]
    .sort_values("confidence", ascending=True, na_position="first")
    .head(30)
)

In [0]:
# ============================================================================
# LLM-Based Topic Assignment (Hybrid: fixed taxonomy + novel-topic discovery)
#
# Per-document, multi-label classification.
#   - Prefers the fixed TOPIC_LIST when a listed topic fits.
#   - Lets the LLM INVENT a new concise topic when nothing fits.
#   - Adds controlled root-cause & impact vocabularies, evidence span,
#     confidence rubric, grounding guardrails.
#   - Normalizes/snaps returned labels to canonical spellings; tracks novel ones.
#
# Databricks Foundation Model API (GPT-5-4 Mini — fast + reliable JSON).
# ============================================================================
import json, time, os, re
import numpy as np, pandas as pd
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI

# ---- Setup client ----
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
client = OpenAI(api_key=token, base_url=f"{host}/serving-endpoints")
MODEL = "topic-modeling-gemma"

# ---- Controlled vocabularies ----
TOPIC_LIST = [
    "Inadvertent unblinding",
    "GMP manufacturing deviation",
    "Labeling and packaging error",
    "Audit finding or inspection observation",
    "Stability testing OOS",
    "Data integrity or GCP issue",
    "Training non-compliance",
    "CAPA overdue or inadequate",
    "Clinical protocol deviation",
    "Vendor or supplier quality issue",
    "Document control or SOP deviation",
    "Equipment or instrument malfunction",
    "Environmental monitoring excursion",
    "Product complaint or adverse event",
    "IT system or CSV issue",
    "Material or component failure",
    "Cleaning or contamination control",
    "Transportation or cold chain deviation",
    "Pharmacovigilance or safety reporting",
    "Regulatory submission or labeling update",
    "Clinical study operations",
]

ROOT_CAUSE_LIST = [
    "Human Error", "Training", "Process/Procedure", "Vendor/CRO",
    "System/Technology (IRT/eCRF)", "Documentation", "Clinical Operations",
    "Laboratory", "Manufacturing/Supply", "Not determinable", "Other",
]

IMPACT_LIST = [
    "Patient Safety", "Data Integrity", "Trial Blinding/Unblinding",
    "GCP Compliance", "Product Quality", "Supply Chain",
    "Documentation/Records", "Not determinable", "None",
]

TOPIC_LIST_STR = "\n".join(f"  {i+1}. {t}" for i, t in enumerate(TOPIC_LIST))
ROOT_CAUSE_STR = " | ".join(ROOT_CAUSE_LIST)
IMPACT_STR     = " | ".join(IMPACT_LIST)

# ---- Prompts ----
SYSTEM_PROMPT = """You are a GxP pharmaceutical quality deviation classifier at Takeda,
specializing in clinical trial deviations and QMS records.
Prefer the provided topic list, but you MAY create a new concise topic when no
listed topic fits. Work ONLY from evidence in the text; if information is not
stated, use "Not determinable" rather than guessing. Do not invent facts,
systems, or root causes. Return ONLY valid JSON, no extra text."""

USER_TEMPLATE = """Classify this deviation event.

STEP 1 - Choose a PRIMARY topic:
  - First try to match one topic from the TOPICS list below.
  - ONLY if no listed topic is a good fit, CREATE a new concise topic
    label (2-5 words) that captures the underlying business process.
STEP 2 - Assign 0-2 SECONDARY topics (listed or newly created, same rule).
STEP 3 - root_cause_category from ROOT CAUSE list.
STEP 4 - regulatory_impact from IMPACT list.
STEP 5 - Quote a short evidence_span justifying the primary topic.
STEP 6 - Score confidence per RUBRIC.
STEP 7 - List any topics you CREATED (not from the list) in "novel_topics".

Rules:
- Prefer listed topics; only invent when clearly necessary.
- Newly created labels must be general business themes, NOT product/study names.
- Keep invented labels concise and reusable.

TOPICS:
{topics}

ROOT CAUSE: {root_causes}

IMPACT: {impacts}

CONFIDENCE RUBRIC:
90-100 = explicit statement in text
70-89  = strong implication
40-69  = partial/indirect evidence
<40    = weak or inferred; text is sparse

Return JSON exactly:
{{"primary": "<listed or new topic>",
  "secondary": ["<topic>", ...],
  "novel_topics": ["<any topic you created>", ...],
  "root_cause_category": "<from ROOT CAUSE>",
  "regulatory_impact": "<from IMPACT>",
  "evidence_span": "<short quote>",
  "confidence": <integer 0-100>,
  "reasoning": "<1 sentence>"}}

Deviation text:
{text}"""

# ---- Canonical lookup + normalizer ----
_TOPIC_CANON = {t.lower(): t for t in TOPIC_LIST}

def _normalize_topic(label):
    """Snap to canonical list if known; else mark as novel. Returns (label, is_known)."""
    if not label or not isinstance(label, str):
        return None, False
    key = label.strip().lower()
    if key in _TOPIC_CANON:
        return _TOPIC_CANON[key], True      # known -> canonical spelling
    return label.strip(), False             # novel -> keep model's wording

# ---- Robust JSON extraction ----
def _extract_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    m = re.search(r'```(?:json)?\s*({.*?})\s*```', text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except json.JSONDecodeError: pass
    m = re.search(r'(\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\})', text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1))
        except json.JSONDecodeError: pass
    raise ValueError(f"No valid JSON found in: {text[:150]}")

# ---- Classify one doc with exponential backoff ----
def classify_doc(pr_id, text, max_retries=4):
    truncated = text[:3000] if len(text) > 3000 else text
    user_msg = USER_TEMPLATE.format(
        topics=TOPIC_LIST_STR, root_causes=ROOT_CAUSE_STR,
        impacts=IMPACT_STR, text=truncated,
    )
    for attempt in range(max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
                max_tokens=350,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
            parsed = _extract_json(raw)

            # Normalize topics: snap known ones, flag novel ones
            prim_label, prim_known = _normalize_topic(parsed.get("primary", "Unknown"))
            sec_norm = [_normalize_topic(s) for s in (parsed.get("secondary", []) or [])]
            secondary = [lbl for lbl, _ in sec_norm if lbl]

            novel = []
            if prim_label and not prim_known:
                novel.append(prim_label)
            novel += [lbl for lbl, known in sec_norm if lbl and not known]
            # merge with model's self-reported novel_topics, de-duped
            novel = sorted(set(novel) | set(parsed.get("novel_topics", []) or []))

            return {
                "pr_id": pr_id,
                "primary": prim_label,
                "primary_is_novel": (prim_label is not None) and (not prim_known),
                "secondary": secondary,
                "novel_topics": novel,
                "root_cause_category": parsed.get("root_cause_category", "Not determinable"),
                "regulatory_impact": parsed.get("regulatory_impact", "Not determinable"),
                "evidence_span": parsed.get("evidence_span", ""),
                "confidence": parsed.get("confidence", None),
                "reasoning": parsed.get("reasoning", ""),
                "raw": raw,
                "error": None,
            }
        except Exception as e:
            if attempt == max_retries:
                return {
                    "pr_id": pr_id, "primary": "ERROR", "primary_is_novel": False,
                    "secondary": [], "novel_topics": [],
                    "root_cause_category": "ERROR", "regulatory_impact": "ERROR",
                    "evidence_span": "", "confidence": None, "reasoning": "",
                    "raw": str(e)[:300], "error": str(e)[:300],
                }
            time.sleep(2 ** attempt)  # 1s, 2s, 4s, 8s

# ---- Load docs ----
llm_pdf = spark.table(EMB_TABLE).select("pr_id", "contextual_retrieval_text_full").toPandas()
llm_pdf["contextual_retrieval_text_full"] = llm_pdf["contextual_retrieval_text_full"].fillna("")
print(f"Documents to classify: {len(llm_pdf):,} | Model: {MODEL} | Topics: {len(TOPIC_LIST)}")

# ---- Run classification (concurrency tuned to respect rate limits) ----
MAX_WORKERS = 20
results, t0 = [], time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(classify_doc, r["pr_id"], r["contextual_retrieval_text_full"]): i
               for i, r in llm_pdf.iterrows()}
    done = 0
    for future in as_completed(futures):
        results.append(future.result())
        done += 1
        if done % 100 == 0:
            el = time.time() - t0; rate = done / el
            print(f"  {done:>5}/{len(llm_pdf)}  ({rate:.1f} docs/s, ETA {(len(llm_pdf)-done)/rate:.0f}s)")
el = time.time() - t0
print(f"\nDone: {len(results):,} docs in {el:.1f}s ({len(results)/max(el,1e-9):.1f} docs/s)")

# ---- Results DataFrame ----
llm_results_df = pd.DataFrame(results)
n_err = llm_results_df["error"].notna().sum()
print(f"Errors: {n_err} ({100*n_err/len(llm_results_df):.1f}%)")

# ---- Multi-label stats ----
llm_results_df["n_topics"] = llm_results_df.apply(
    lambda r: 1 + len(r["secondary"]) if r["primary"] != "ERROR" else 0, axis=1)

# ---- Rollups for dashboard ----
print("\n" + "="*80)
print("PRIMARY TOPIC DISTRIBUTION")
print("="*80)
print(llm_results_df["primary"].value_counts().head(30).to_string())

print("\n" + "="*80)
print("ROOT CAUSE DISTRIBUTION")
print("="*80)
print(llm_results_df["root_cause_category"].value_counts().to_string())

print("\n" + "="*80)
print("REGULATORY IMPACT DISTRIBUTION")
print("="*80)
print(llm_results_df["regulatory_impact"].value_counts().to_string())

# ---- Confidence health check ----
conf = pd.to_numeric(llm_results_df["confidence"], errors="coerce")
print("\n" + "="*80)
print("CONFIDENCE")
print("="*80)
print(f"  mean={conf.mean():.1f}  median={conf.median():.1f}  <40 (needs review): {(conf<40).sum()}")

# ---- Multi-label summary ----
print("\n" + "="*80)
print("MULTI-LABEL STATS")
print("="*80)
for k in [1, 2, 3]:
    print(f"  {k} topic(s): {(llm_results_df['n_topics']==k).sum()}")
print(f"  4+ topics:  {(llm_results_df['n_topics']>=4).sum()}")

# ---- Novel (LLM-invented) topics: taxonomy-expansion candidates ----
novel_counter = Counter()
for _, r in llm_results_df.iterrows():
    novel_counter.update(r["novel_topics"] or [])

print("\n" + "="*80)
print("NOVEL TOPICS (not in TOPIC_LIST)")
print("="*80)
print(f"Rows using a novel PRIMARY topic: {llm_results_df['primary_is_novel'].sum()}")
if novel_counter:
    for topic, cnt in novel_counter.most_common(30):
        print(f"  {cnt:>4}  {topic}")
else:
    print("  (none — every record matched the existing taxonomy)")

# ---- Taxonomy coverage ----
valid = (llm_results_df["primary"] != "ERROR").sum()
known_primary = (~llm_results_df["primary_is_novel"] &
                 (llm_results_df["primary"] != "ERROR")).sum()
print(f"\nTaxonomy coverage (primary): {known_primary}/{valid} "
      f"({100*known_primary/max(valid,1):.1f}%)")

# ---- Display: low-confidence first for QA ----
display(
    llm_results_df[["pr_id", "primary", "primary_is_novel", "secondary",
                    "novel_topics", "root_cause_category", "regulatory_impact",
                    "confidence", "n_topics", "evidence_span", "reasoning"]]
    .sort_values("confidence", ascending=True, na_position="first")
    .head(30)
)

In [0]:
# Quick check: what were the errors from the previous run?
errors = llm_results_df[llm_results_df["error"].notna()]
print(f"Total errors: {len(errors)} / {len(llm_results_df)}")
print(f"\nFirst 5 unique error messages:")
for i, msg in enumerate(errors["raw"].unique()[:5]):
    print(f"\n--- Error {i+1} ---")
    print(msg[:500])

In [0]:
# ============================================================================
# BGE-M3 full-tier vs core-tier — Topic Modeling Comparison
# Both columns are 1024-dim BGE-M3 vectors from the SAME model, differing only
# in how much context was embedded:
#   • embedding       ← BGE-M3 over contextual_retrieval_text_full  (full context)
#   • core_embedding  ← BGE-M3 over contextual_retrieval_text_core  (core only)
# Same BERTopic config (best UMAP params, same HDBSCAN, same c-TF-IDF).
# ============================================================================
import numpy as np, pandas as pd, plotly.express as px
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# ---- Load both embedding tiers from the same table ----
comp_pdf = (
    spark.table(EMB_TABLE)
    .select("pr_id", "source_free_text_full", "embedding", "core_embedding")
    .toPandas()
)
comp_pdf["source_free_text_full"] = comp_pdf["source_free_text_full"].fillna("")

# Clean docs for c-TF-IDF (reuse _strip_for_tfidf from Cell 2)
comp_docs = [_strip_for_tfidf(t) for t in comp_pdf["source_free_text_full"].tolist()]

embs_full = np.array(comp_pdf["embedding"].tolist(), dtype=np.float32)
embs_core = np.array(comp_pdf["core_embedding"].tolist(), dtype=np.float32)

print(f"Loaded {len(comp_docs):,} docs")
print(f"  BGE-M3 full tier (embedding):      {embs_full.shape}")
print(f"  BGE-M3 core tier (core_embedding): {embs_core.shape}")

# ---- Shared config (best UMAP params from grid search) ----
BEST_NN, BEST_NC, BEST_MD = 5, 5, 0.0

STRUCTURAL_STOPS = [
    "event", "title", "description", "impact", "assessment", "quality", "final",
    "root", "cause", "category", "subcategory", "action", "text",
    "resolved", "references", "canonical", "vendor", "external", "supplier",
    "country", "acronym", "definitions", "cro", "system", "name", "document",
    "device", "enrichment", "clinical", "id", "type", "number",
]
custom_stops = list(ENGLISH_STOP_WORDS) + STRUCTURAL_STOPS

def _fit_bertopic(embeddings, label):
    """Fit BERTopic on given embeddings and return (topics, model, metrics)."""
    model = BERTopic(
        embedding_model=None,
        umap_model=UMAP(n_neighbors=BEST_NN, n_components=BEST_NC, min_dist=BEST_MD,
                        metric="cosine", random_state=42),
        hdbscan_model=HDBSCAN(min_cluster_size=10, min_samples=5,
                              metric="euclidean", cluster_selection_method="eom",
                              prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words=custom_stops, min_df=2, ngram_range=(1, 2)),
        calculate_probabilities=True, verbose=False,
    )
    topics_i, probs_i = model.fit_transform(comp_docs, embeddings=embeddings)

    n_topics = len(set(topics_i)) - (1 if -1 in topics_i else 0)
    outlier_pct = 100 * (np.array(topics_i) == -1).mean()
    avg_prob = probs_i.max(axis=1).mean() if probs_i is not None else 0.0
    all_words = []
    for t in set(topics_i):
        if t == -1: continue
        all_words.extend([w for w, _ in model.get_topic(t)[:5]])
    diversity = len(set(all_words)) / max(len(all_words), 1)

    return topics_i, model, {
        "Model": label, "Topics": n_topics,
        "Outlier %": round(outlier_pct, 1),
        "Avg Prob": round(avg_prob, 4),
        "Diversity": round(diversity, 4),
    }

print("\nFitting BGE-M3 (full tier)...")
topics_full, model_full, m_full = _fit_bertopic(embs_full, "BGE-M3 (full)")

print("Fitting BGE-M3 (core tier)...")
topics_core, model_core, m_core = _fit_bertopic(embs_core, "BGE-M3 (core)")

# ---- Comparison metrics table ----
comp_table = pd.DataFrame([m_full, m_core])
print("\n" + "="*65)
print("TIER COMPARISON  (same UMAP/HDBSCAN params, same c-TF-IDF text)")
print("="*65)
display(comp_table)

# ---- Clustering agreement ----
ari = adjusted_rand_score(topics_full, topics_core)
nmi = normalized_mutual_info_score(topics_full, topics_core)
print(f"\nClustering Agreement (full vs core tier):")
print(f"  Adjusted Rand Index:           {ari:.4f}  (1.0 = identical, 0.0 = random)")
print(f"  Normalized Mutual Information:  {nmi:.4f}  (1.0 = identical partitions)")

# ---- Side-by-side 3D UMAP scatter ----
umap_3d_viz = UMAP(n_neighbors=BEST_NN, n_components=3, min_dist=BEST_MD,
                   metric="cosine", random_state=42)

coords_full = umap_3d_viz.fit_transform(embs_full)
coords_core = umap_3d_viz.fit_transform(embs_core)

desc_preview = [d[:250] + ("..." if len(d) > 250 else "") for d in comp_docs]

# BGE-M3 full-tier plot
plot_full = pd.DataFrame({
    "UMAP-1": coords_full[:, 0], "UMAP-2": coords_full[:, 1], "UMAP-3": coords_full[:, 2],
    "topic": [str(t) for t in topics_full],
    "pr_id": comp_pdf["pr_id"].values,
    "event_description": desc_preview,
})
fig1 = px.scatter_3d(
    plot_full, x="UMAP-1", y="UMAP-2", z="UMAP-3", color="topic",
    hover_data={"pr_id": True, "event_description": True, "UMAP-1": False, "UMAP-2": False, "UMAP-3": False},
    title=f"BGE-M3 (full) — {m_full['Topics']} topics, {m_full['Outlier %']}% outlier",
    opacity=0.7, height=650,
)
fig1.update_traces(marker_size=3)
fig1.show()

# BGE-M3 core-tier plot
plot_core = pd.DataFrame({
    "UMAP-1": coords_core[:, 0], "UMAP-2": coords_core[:, 1], "UMAP-3": coords_core[:, 2],
    "topic": [str(t) for t in topics_core],
    "pr_id": comp_pdf["pr_id"].values,
    "event_description": desc_preview,
})
fig2 = px.scatter_3d(
    plot_core, x="UMAP-1", y="UMAP-2", z="UMAP-3", color="topic",
    hover_data={"pr_id": True, "event_description": True, "UMAP-1": False, "UMAP-2": False, "UMAP-3": False},
    title=f"BGE-M3 (core) — {m_core['Topics']} topics, {m_core['Outlier %']}% outlier",
    opacity=0.7, height=650,
)
fig2.update_traces(marker_size=3)
fig2.show()

# ---- Top-5 keywords comparison for shared top topics ----
print("\nTop 5 keywords per tier (first 10 topics):")
print(f"{'Topic':<8} {'BGE-M3 (full)':<55} {'BGE-M3 (core)'}")
print("-" * 120)
for t in range(min(10, m_full["Topics"], m_core["Topics"])):
    kw_full = ", ".join(w for w, _ in model_full.get_topic(t)[:5])
    kw_core = ", ".join(w for w, _ in model_core.get_topic(t)[:5])
    print(f"{t:<8} {kw_full:<55} {kw_core}")

In [0]:
# ============================================================================
# Inspect Zero-Shot BGE-M3 topics + attach labels back to pr_id
# ============================================================================
import numpy as np

# Top keywords per topic (from BGE-M3 zero-shot model)
for t in sorted(set(zs_bge_topics)):
    if t == -1:
        continue
    words = ", ".join(w for w, _ in zs_bge_model.get_topic(t)[:8])
    print(f"Topic {t:>2}: {words}")

# Map topic + representative keywords back onto each deviation
pdf["topic"] = zs_bge_topics
pdf["topic_prob"] = zs_bge_probs.max(axis=1) if zs_bge_probs is not None else np.nan

# Write back to Delta so downstream can join on pr_id
CATALOG = "us_gmsgq_dev"
ALYT = "gms_us_alyt"
out_df = spark.createDataFrame(
    pdf[["pr_id", "topic", "topic_prob"]].astype({"pr_id": str, "topic": int})
)
(out_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{ALYT}.deviation_topics"))
print(f"✓ wrote {CATALOG}.{ALYT}.deviation_topics")

# ---- Write 3-column topic assignments (primary/secondary/tertiary labels) ----
# Uses zs_bge_multi_df built by the multi-label section of the BGE-M3 cell.
topic_assign_pdf = zs_bge_multi_df[["pr_id", "all_labels"]].copy()
topic_assign_pdf["primary_topic"] = topic_assign_pdf["all_labels"].apply(
    lambda labels: labels[0] if labels and len(labels) > 0 else None
)
topic_assign_pdf["secondary_topic"] = topic_assign_pdf["all_labels"].apply(
    lambda labels: labels[1] if labels and len(labels) > 1 else None
)
topic_assign_pdf["tertiary_topic"] = topic_assign_pdf["all_labels"].apply(
    lambda labels: labels[2] if labels and len(labels) > 2 else None
)

assign_sdf = spark.createDataFrame(
    topic_assign_pdf[["pr_id", "primary_topic", "secondary_topic", "tertiary_topic"]]
)
(assign_sdf.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{ALYT}.deviation_topic_assignments"))
print(f"✓ wrote {CATALOG}.{ALYT}.deviation_topic_assignments  "
      f"({len(topic_assign_pdf):,} events × 3 label columns)")

## Multi-Label Topic Assignment Schema

Each deviation event is assigned **up to 3 topic labels**, ranked by confidence:

| Column | Meaning |
| --- | --- |
| **Primary** | The single best-fit topic for this event. Always exactly one. |
| **Secondary** | The next most relevant topic, if the event spans multiple themes. Exactly one or empty. |
| **tertiary** | Any further topics detected beyond the top two. May contain multiple labels (pipe-separated) or be empty. Most events will have none. |

**Why "tertiary" can have multiple labels:**  
The BGE-M3 model uses a sliding window over the document text, computing topic relevance for each passage independently. A long event describing several distinct issues (e.g., a manufacturing deviation in one paragraph, an overdue CAPA in another, and a documentation gap in a third) will surface all of them. Rather than arbitrarily discarding the 4th or 5th detected theme, we preserve them here for transparency. The LLM is prompt-constrained to return at most 2 secondary topics, so its "additional" column will have at most one entry.

In [0]:
# ============================================================================
# CROSS-MODEL LABEL COMPARISON
#
# For a sample of events, show what label each model assigned.
# Models: Zero-Shot BGE-M3, LLM (GPT-5-4-mini)
# ============================================================================
import pandas as pd
import numpy as np

# ---- Build label lookup for Zero-Shot BGE-M3 ----
zs_bge_labels = topic_labels_map  # built in the BGE-M3 zero-shot cell

# ---- Assemble comparison DataFrame ----
comparison = pd.DataFrame({
    "pr_id": pdf["pr_id"].values,
    "text": docs,
    "ZS_primary": [zs_bge_labels.get(t, f"T{t}") for t in zs_bge_topics],
})

# ---- Add SECONDARY + ADDITIONAL topics ----
# Primary = best-fit single topic
# Secondary = next-best single topic
# Additional = any remaining topics detected (pipe-separated; may be empty)

# Zero-Shot BGE-M3: split all_labels into secondary (one) and tertiary (rest)
bge_multi = zs_bge_multi_df.set_index("pr_id")["all_labels"].apply(
    lambda labels: pd.Series({
        "ZS_secondary": labels[1] if labels is not None and len(labels) > 1 else "",
        "ZS_tertiary": labels[2] if labels is not None and len(labels) > 2 else "",
    })
)
comparison = comparison.merge(bge_multi, left_on="pr_id", right_index=True, how="left")

# LLM: secondary[0] = secondary, secondary[1:] = tertiary (pipe-separated)
llm_labels = llm_results_df.set_index("pr_id")[["primary", "secondary"]].copy()
llm_labels["LLM_secondary"] = llm_labels["secondary"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else ""
)
llm_labels["LLM_tertiary"] = llm_labels["secondary"].apply(
    lambda x: x[1] if isinstance(x, list) and len(x) > 1 else ""
)
llm_labels = llm_labels.rename(columns={"primary": "LLM_primary"})[["LLM_primary", "LLM_secondary", "LLM_tertiary"]]
comparison = comparison.merge(llm_labels, left_on="pr_id", right_index=True, how="left")

# ---- Show a random sample of 20 events with all labels ----
print("="*100)
print("CROSS-MODEL LABEL COMPARISON  (20 random events)")
print("="*100)
print(f"Total events: {len(comparison):,}")
print()

sample = comparison.sample(n=20, random_state=42)
display(
    sample[[
        "pr_id", "text",
        "ZS_primary", "ZS_secondary", "ZS_tertiary",
        "LLM_primary", "LLM_secondary", "LLM_tertiary",
    ]]
)
